In [0]:
import json, os
d = "/Volumes/workspace/landing/raw/incremental/drift-test/test__repo/issues"
os.makedirs(d, exist_ok=True)
good = {"repo_name": "test/repo", "entity_type": "issues", "id": 900000001, "number": 1,
        "title": "good row", "state": "open", "user": {"login": "u1", "id": 1, "type": "User"},
        "labels": [], "comments": 2, "created_at": "2026-10-15T10:00:00Z",
        "updated_at": "2026-10-15T10:00:00Z", "closed_at": None, "pull_request": None,
        "new_field_from_github": "extra"}
bad = dict(good, id=900000002, number=2, title="bad row", comments="many")
open(d + "/page_0001.jsonl", "w").write(json.dumps(good) + "\n" + json.dumps(bad) + "\n")
print("created", os.listdir(d))

created ['page_0001.jsonl']


In [0]:
dbutils.notebook.run("./02_raw_to_bronze", 600,
    {"batch_date": "drift-test", "load_type": "incremental", "entity": "issues", "raw_root": "/Volumes/workspace/landing/raw"})

In [0]:
%sql
SELECT entity, batch_date, source_file, raw_record
FROM workspace.bronze.quarantine
WHERE batch_date = 'drift-test'

In [0]:
%sql
SELECT layer, entity, parameter, status, rows_inserted, error_message
FROM workspace.ops.pipeline_execution_logs
WHERE parameter LIKE 'incremental/drift-test/%'

In [0]:
%sql
DELETE FROM workspace.bronze.issues WHERE repo_name = 'test/repo'

In [0]:
%sql
DELETE FROM workspace.bronze.quarantine WHERE batch_date = 'drift-test'

In [0]:
dbutils.fs.rm("/Volumes/workspace/landing/raw/incremental/drift-test", True)